# Run Talent Agent with the configured defaults

Run the cells from top to bottom to start a normal Talent Agent search. The notebook does not override the project configuration: the CLI reads `config/search.json`, loads credentials from the project `.env`, and uses the default candidate CV and model unless they are configured otherwise. Re-running this cell resumes today's incomplete run when checkpoint files exist, instead of repeating completed work.

## Prerequisites

- Install the project dependencies in a Python environment and select that environment as this notebook's kernel (for example, `talent-agent/.venv`).
- Create `talent-agent/.env` with `OPENROUTER_API_KEY`, `APIFY_API_TOKEN`, and `APIFY_ACTOR_ID` configured. Do not put secrets in this notebook.
- Ensure the default candidate CV exists at `candidate/master_cv.docx`.
- Review `config/search.json` before running; search and model requests may incur charges or use API quota.

In [ ]:
from pathlib import Path
import subprocess
import sys

# The notebook lives in the project's notebooks/ directory; also support launching it from the repository root.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Could not locate talent-agent/pyproject.toml from the notebook working directory")

print(f"Project: {PROJECT_ROOT}")
print(f"Python:  {sys.executable}")
assert (PROJECT_ROOT / "candidate" / "master_cv.docx").is_file(), "Default CV not found: candidate/master_cv.docx"
assert (PROJECT_ROOT / ".env").is_file(), "Project .env not found; configure credentials before running"


In [ ]:
import json
from datetime import date

# Continue today's interrupted run when it has checkpoints; otherwise start a fresh default search.
config_path = PROJECT_ROOT / "config" / "search.json"
search_config = json.loads(config_path.read_text(encoding="utf-8"))
output_settings = search_config["output"]
output_root = Path(output_settings["root_dir"])
if not output_root.is_absolute():
    output_root = PROJECT_ROOT / output_root
if output_settings["dated_subfolders"]:
    run_directory = output_root / date.today().strftime(output_settings["date_format"])
else:
    run_directory = output_root
state_path = run_directory / "agent_outputs" / "run_state.json"
resume_incomplete_run = False
if state_path.is_file():
    run_state = json.loads(state_path.read_text(encoding="utf-8"))
    stage_statuses = run_state.get("stages", {})
    resume_incomplete_run = bool(stage_statuses) and any(
        status != "complete" for status in stage_statuses.values()
    )

command = [sys.executable, "-m", "talent_agent.cli", "run"]
if resume_incomplete_run:
    command.extend(["--resume", str(run_directory)])
    print(f"Resuming interrupted run: {run_directory}")
else:
    command.append("--search")
    print("Starting a new search using config/search.json defaults.")

completed = subprocess.run(command, cwd=PROJECT_ROOT, check=True, text=True)
print("Talent Agent finished successfully.")


In [ ]:
from IPython.display import Markdown, display

OUTPUT_ROOT = output_root
run_directories = sorted(
    (path for path in OUTPUT_ROOT.iterdir() if path.is_dir()),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
) if OUTPUT_ROOT.is_dir() else []

if not run_directories:
    print("No output run directory was found.")
else:
    latest_run = run_directories[0]
    print(f"Latest run: {latest_run}")
    for artifact in sorted(latest_run.rglob("*")):
        if artifact.is_file():
            relative_path = artifact.relative_to(PROJECT_ROOT)
            display(Markdown(f"- [{relative_path}]({artifact.resolve().as_uri()})"))
